# Gold Layer - Vehicle Efficiency

## Objective
This notebook creates an analytical gold table from the silver vehicle fuel economy dataset.

## Main goals
- aggregate vehicle efficiency metrics into a business-friendly analytical table
- support analysis of fuel economy, emissions, and fuel cost trends
- provide a simplified view of vehicle performance by make, model, and year

## Notes
The gold layer focuses on analytical value, so this table contains aggregated efficiency metrics instead of detailed raw vehicle records.

In [0]:
from pyspark.sql import functions as F

## Imports and configuration

Define the source and target tables used in this notebook.

In [0]:
catalog_name = "automotive"
silver_schema = "silver"
gold_schema = "gold"

silver_fuel_economy_table = f"{catalog_name}.{silver_schema}.vehicle_fuel_economy"
gold_vehicle_efficiency_table = f"{catalog_name}.{gold_schema}.vehicle_efficiency"

## Read silver table

Load the cleaned vehicle fuel economy dataset from the silver layer.

In [0]:
df_fuel_economy_silver = spark.table(silver_fuel_economy_table)

## Build analytical aggregation

Aggregate fuel economy and emissions metrics by make, model, and vehicle year.

In [0]:
df_gold_vehicle_efficiency = (
    df_fuel_economy_silver
    .groupBy("make", "model", "vehicle_year")
    .agg(
        F.avg("comb08").alias("avg_comb08"),
        F.avg("city08").alias("avg_city08"),
        F.avg("highway08").alias("avg_highway08"),
        F.avg("co2").alias("avg_co2"),
        F.avg("fuelcost08").alias("avg_fuelcost08")
    )
)

## Reorder columns

Move the most relevant analytical fields to the front.

In [0]:
preferred_column_order = [
    "make",
    "model",
    "vehicle_year",
    "avg_comb08",
    "avg_city08",
    "avg_highway08",
    "avg_co2",
    "avg_fuelcost08"
]

final_columns = [c for c in preferred_column_order if c in df_gold_vehicle_efficiency.columns] + [
    c for c in df_gold_vehicle_efficiency.columns if c not in preferred_column_order
]

df_gold_vehicle_efficiency = df_gold_vehicle_efficiency.select(*final_columns)

## Persist gold table

Save the aggregated vehicle efficiency table into the gold layer as a Delta table.

In [0]:
(
    df_gold_vehicle_efficiency.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_vehicle_efficiency_table)
)

## Final note

This gold table provides a simplified analytical view of fuel economy, emissions, and fuel cost trends by vehicle, supporting broader automotive industry analysis.

## Data preview

Display statements are included for demonstration purposes to provide a quick visual inspection of the data in each layer.

These previews are intended to support the assessment by allowing easy validation of the transformations applied throughout the pipeline.

In [0]:
display(spark.table(gold_vehicle_efficiency_table))

make,model,vehicle_year,avg_comb08,avg_city08,avg_highway08,avg_co2,avg_fuelcost08
PLYMOUTH,COLT VISTA,1993,20.75,18.75,24.125,null,2125.0
MAZDA,B2200/B2600I,1993,20.0,18.166666666666668,23.166666666666668,null,2200.0
CHEVROLET,ASTRO 2WD (CARGO),1993,17.0,15.0,20.5,null,3000.0
GEO,TRACKER CONVERTIBLE 2WD,1993,22.0,21.0,23.0,null,2000.0
CHEVROLET,BERETTA,1994,21.833333333333332,19.166666666666668,27.5,null,2133.3333333333335
ROLLS-ROYCE,BROOKLANDS AND (LWB),1994,11.0,9.0,14.0,null,5250.0
TOYOTA,COROLLA WAGON,1994,25.5,23.0,30.0,null,1725.0
DODGE,DAKOTA PICKUP 4WD,1994,14.5,13.25,17.25,null,3037.5
PONTIAC,TRANS SPORT 2WD,1994,18.0,16.0,22.0,null,2450.0
DODGE,DAKOTA CAB CHASSIS 2WD,1994,13.25,12.25,15.0,null,3312.5
